# Simple RAG (Retrieval-Augmented Generation) System for CSV Files

In [20]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set")

HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")

if not HUGGINGFACE_API_KEY:
    raise ValueError("HUGGINGFACE_API_KEY is not set")

GROQ_MODEL = os.getenv("GROQ_MODEL")

HUGGINGFACE_MODEL = os.getenv("HUGGINGFACE_MODEL")

# Embedding

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu",
    },
    encode_kwargs={
        "normalize_embeddings": True,
    },
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6033.45it/s]


In [23]:
embedding = embedding_model.embed_query(
    "Which company does John Doe work for?"
)

print("Embedding dimensions:", len(embedding))

Embedding dimensions: 384


# Loader

In [15]:
from langchain_community.document_loaders import CSVLoader

def load_csv(file_path: str):
    loader = CSVLoader(
        file_path=file_path,
        encoding="utf-8",
    )

    documents = loader.load()

    return documents

In [25]:
documents = load_csv("../data/customers.csv")
print(documents[0].page_content)

first_name: John
last_name: Doe
company: Acme Inc
email: john@example.com
phone: 9876543210
city: New York
country: USA


# Vector Store

In [26]:
from langchain_chroma import Chroma

CHROMA_PATH = "../chroma_db"
COLLECTION_NAME = "customer_rag"

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=CHROMA_PATH,
)

print("ChromaDB initialized")

ChromaDB initialized


In [27]:
vector_store.add_documents(documents)

print(f"Added {len(documents)} documents to ChromaDB")

Added 8 documents to ChromaDB


# Create Retriever

In [28]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
    },
)

# Test retrieval

In [29]:
question = "Which company does John Doe work for?"

retrieved_docs = retriever.invoke(question)

print("Retrieved documents:", len(retrieved_docs))

Retrieved documents: 4


In [30]:
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n========== DOCUMENT {i} ==========")
    print(doc.page_content)


========== DOCUMENT 1 ==========
first_name: John
last_name: Doe
company: Acme Inc
email: john@example.com
phone: 9876543210
city: New York
country: USA

========== DOCUMENT 2 ==========
first_name: Bob
last_name: Johnson
company: Google
email: bob@example.com
phone: 9876543212
city: California
country: USA

========== DOCUMENT 3 ==========
first_name: David
last_name: Brown
company: Amazon
email: david@example.com
phone: 9876543214
city: London
country: UK

========== DOCUMENT 4 ==========
first_name: Michael
last_name: Davis
company: Meta
email: michael@example.com
phone: 9876543215
city: California
country: USA


# Initialize Groq

In [31]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=GROQ_MODEL,
    temperature=0,
)

# Create RAG prompt

In [32]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a helpful customer information assistant.

Answer the user's question using ONLY the provided context.

Rules:
1. Do not invent information.
2. Do not use outside knowledge.
3. If the answer is not present in the context, say:
   "I could not find this information in the customer data."
4. Keep the answer concise and clear.

Context:
{context}
""",
        ),
        (
            "human",
            "{question}",
        ),
    ]
)

# Create RAG function

In [33]:
def ask_rag(question: str, k: int = 4):
    
    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)
    
    # Convert documents into context
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )
    
    # Create chain
    chain = prompt | llm
    
    # Generate answer
    response = chain.invoke(
        {
            "context": context,
            "question": question,
        }
    )
    
    return response.content, retrieved_docs

# Ask a question

In [36]:
question = "Which company does John Doe work for?"

answer, docs = ask_rag(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

QUESTION:
Which company does John Doe work for?

ANSWER:
John Doe works for Acme Inc.


# Show retrieved sources

In [37]:
print("RETRIEVED SOURCES:")

for i, doc in enumerate(docs, 1):
    print(f"\n========== SOURCE {i} ==========")
    print(doc.page_content)

RETRIEVED SOURCES:

========== SOURCE 1 ==========
first_name: John
last_name: Doe
company: Acme Inc
email: john@example.com
phone: 9876543210
city: New York
country: USA

========== SOURCE 2 ==========
first_name: Bob
last_name: Johnson
company: Google
email: bob@example.com
phone: 9876543212
city: California
country: USA

========== SOURCE 3 ==========
first_name: David
last_name: Brown
company: Amazon
email: david@example.com
phone: 9876543214
city: London
country: UK

========== SOURCE 4 ==========
first_name: Michael
last_name: Davis
company: Meta
email: michael@example.com
phone: 9876543215
city: California
country: USA


In [38]:
question = "Where does Alice Smith live?"

answer, docs = ask_rag(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

QUESTION:
Where does Alice Smith live?

ANSWER:
London, UK


# Interactive chat

In [40]:
while True:
    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() in ["exit", "quit"]:
        print("Chat ended.")
        break

    answer, docs = ask_rag(question)

    print("\nAI:")
    print(answer)


AI:
Customers from London:
- David Brown
- Alice Smith
Chat ended.
